# Alerts Pipeline — End-to-End QC

Runs all five pipeline stages (detect → score → facts → investigate) with inspection cells between each stage.

| Cell | Stage | What happens |
|------|-------|--------------|
| 1 | Config | Set analysis date, optional store filter, composite score weights |
| 2 | Session | Connect to Snowflake, load typed config |
| 3 | Data load — OSA | OSA + hierarchy joined, feedback cached |
| 4 | Data load — WMD | Markdown log, sales, wastage, item/store hierarchy |
| 5–6 | **Stage 1 — Detect** | Run all detectors, inspect anomalies by signal type |
| 7–9 | **Stage 2 — Score** | Score signals, inspect composite dimension breakdown |
| 10–11 | **Stage 3 — Facts** | Enrich signals, inspect facts per signal type |
| 12–13 | **Stage 4 — Investigate** | Generate narratives, inspect per signal type |
| 14 | Full card preview | Read one complete alert card per signal type |

In [1]:
# ── Date window ────────────────────────────────────────────────────────────────
# Set ANALYSIS_DATE_STR to the last day of the week you want to analyse.
ANALYSIS_DATE_STR = '2026-03-20'

# ── Client config (None = base defaults; 'eastofengland' = client override) ────
CLIENT = None

# ── Optional: restrict to one store for a faster test run ─────────────────────
QC_STORE_ID = None   # e.g. 507 — set to None for all stores

# ── Fact-gathering parameters ─────────────────────────────────────────────────
PROJECTION_WEEKS = 4      # cost-if-not-fixed projection horizon
DRILLDOWN_TOP_N  = 3      # top-N PPGs in drilldown lists

# ── Composite score config for QC ─────────────────────────────────────────────
# Leave COMPOSITE_WEIGHTS empty ({}) to use raw projected_cost scoring.
# Populate to see composite dimension breakdown in Stage 2 cells.
COMPOSITE_WEIGHTS = {
    'financial_severity':      0.50,
    'vs_control_benchmark':    0.25,
    'vs_historical_benchmark': 0.10,
    'thematic':                0.05,
    'historical_recency':      0.05,
    'special_cases':           0.05,
}
COST_CEILINGS = {
    'OSA':                  5000.0,
    'COMPLIANCE':           5000.0,
    'WASTE':                2000.0,
    'MARKDOWN_OVERRIDE':    2000.0,
    'MARKDOWN_SELLTHROUGH': 2000.0,
    'MARKDOWN_SKU':         1000.0,
}
BENCHMARK_CEILINGS = {
    'OSA': 0.10, 'COMPLIANCE': 0.10, 'WASTE': 0.15,
    'MARKDOWN_OVERRIDE': 0.15, 'MARKDOWN_SELLTHROUGH': 0.15,
}
WOW_CEILINGS = {
    'OSA': 0.10, 'COMPLIANCE': 0.10, 'WASTE': 0.15,
    'MARKDOWN_OVERRIDE': 0.15, 'MARKDOWN_SELLTHROUGH': 0.15,
}

import datetime

analysis_date   = datetime.date.fromisoformat(ANALYSIS_DATE_STR)
THIS_WEEK_END   = str(analysis_date)
THIS_WEEK_START = str(analysis_date - datetime.timedelta(days=6))
LAST_WEEK_END   = str(analysis_date - datetime.timedelta(days=7))
LAST_WEEK_START = str(analysis_date - datetime.timedelta(days=13))
LOOKBACK_START  = str(analysis_date - datetime.timedelta(days=69))  # ~10 weeks
WMD_LOOKBACK    = str(analysis_date - datetime.timedelta(days=27))  # 4 weeks for SKU detectors

print(f'Analysis date : {THIS_WEEK_END}')
print(f'This week     : {THIS_WEEK_START} – {THIS_WEEK_END}')
print(f'Last week     : {LAST_WEEK_START} – {LAST_WEEK_END}')
print(f'Lookback      : {LOOKBACK_START} – {THIS_WEEK_END}')
print(f'WMD window    : {WMD_LOOKBACK} – {THIS_WEEK_END}')
print(f'Client        : {CLIENT or "base defaults"}')
print(f'Store filter  : {QC_STORE_ID or "all stores"}')
print(f'Composite     : {"enabled" if COMPOSITE_WEIGHTS else "disabled (raw projected_cost)"}')

Analysis date : 2026-03-20
This week     : 2026-03-14 – 2026-03-20
Last week     : 2026-03-07 – 2026-03-13
Lookback      : 2026-01-10 – 2026-03-20
WMD window    : 2026-02-21 – 2026-03-20
Client        : base defaults
Store filter  : all stores
Composite     : enabled


In [2]:
import sys, os, logging, warnings

_root = os.path.abspath(os.path.join('..', '..', '..'))
if _root not in sys.path:
    sys.path.insert(0, _root)

logging.getLogger('snowflake.connector').setLevel(logging.ERROR)
logging.getLogger('snowflake.snowpark').setLevel(logging.ERROR)
warnings.filterwarnings('ignore', category=UserWarning, module='snowflake')

from store_insight.pipeline.config.loader import load_config
from store_insight.pipeline.pipeline.session import create_session

cfg = load_config(CLIENT)
print(f'Config loaded: client={cfg.client_name}')

Config loaded: client=default


In [3]:
session = create_session(cfg)

 pip install snowflake-connector-python[secure-local-storage]


Initiating login request with your identity provider. A browser window should have opened for you to complete the login. If you can't see it, check existing browser windows, or your OS settings. Press CTRL+C to abort and try again...
Going to open: https://login.microsoftonline.com/93a31d5d-1a5d-433f-baa0-138549cd2108/saml2?SAMLRequest=nZJRb5swFIX%2FCvKeMTYkWWIlqVhRtEjdmja0U%2FfmgkmsGJv4mtLu18%2BQROoe2odJCCFz7v2O77nzq9daBS%2FCgjR6gSgmKBC6MKXUuwV6yFfhFAXguC65Mlos0JsAdLWcA69Vw9LW7fW9OLYCXOAbaWD9jwVqrWaGgwSmeS2AuYJt0x83LMaEcQBhncehc0kJ0rP2zjUsirquw12Cjd1FMSEkIrPIq3rJF%2FQO0XzOaKxxpjDqUvLq7%2FQBgkZk1CO8whM258JvUp9G8Bnl%2BSQC9j3PN%2BHmdpujIL3c7tpoaGtht8K%2ByEI83N%2BcDIB3ADWdJGSE20MIxs8Q8z%2BtFRi06SrFD6IwddM63xr7r6gSZaTMTvqBrbMFag6yPHYzlTm%2Bv7NP09SMjnd1Qyf7jB71sXlMZ1U2WR3Uzv3iT4cCBY%2BXeOM%2B3jVAK9a6D9X5IxJPQuIfmtMxSyijE%2Fw1Jr9RkPlQpeZuqLw4H3zgWhbWgKmc0UpqMbicJTyh5bgMKfevUZJU4TPnJKTJdDyaFWVMyTTqo4vRaX3YYMQu%2F2so8%2Bh9i%2FM6%2FvQJrbONUbJ4C1bG1tx9HCDFdDiRZVgNUiZqLlVallYA%2BCCVMt21Fdz5rXe2FS

 pip install snowflake-connector-python[secure-local-storage]


## Data Load — OSA + Feedback

In [4]:
from snowflake.snowpark import functions as F

OSA_TABLE            = cfg.tables.osa
HIERARCHY_TABLE      = cfg.tables.hierarchy
FEEDBACK_TABLE       = cfg.tables.feedback
MARKDOWN_LOG_TABLE   = cfg.tables.markdown_log
MARKDOWN_SALES_TABLE = cfg.tables.markdown_sales
WASTAGE_TABLE        = cfg.tables.wastage
WMD_HIERARCHY_TABLE  = cfg.tables.wmd_hierarchy
STORE_HIER_TABLE     = cfg.tables.store_hierarchy


print(f'OSA table      : {OSA_TABLE}')
print(f'Hierarchy table: {HIERARCHY_TABLE}')
print(f'Feedback table : {FEEDBACK_TABLE}')
print(f'Markdown log   : {MARKDOWN_LOG_TABLE}')
print(f'Markdown sales : {MARKDOWN_SALES_TABLE}')
print(f'Wastage        : {WASTAGE_TABLE}')
print(f'WMD hierarchy  : {WMD_HIERARCHY_TABLE}')
print(f'Store hierarchy: {STORE_HIER_TABLE}')


osa_raw = (
    session.table(OSA_TABLE)
    .filter(F.col('DATE') >= LOOKBACK_START)
    .filter(F.col('DATE') <= THIS_WEEK_END)
    .select(
        F.col('STORE_ID'),
        F.col('ITEM_CODE'),
        F.col('ACTUAL_SALES_VOLUME'),
        F.col('LOST_SALES_VOLUME'),
        F.col('LOST_SALES_VALUE'),
        F.col('DATE'),
        F.col('DEPARTMENT_ID'),
    )
)

item = (
    session.table(HIERARCHY_TABLE)
    .select(
        F.col('ITEM_CODE'),
        F.col('ITEM_HIERARCHY_1'),
        F.col('ITEM_HIERARCHY_2'),
        F.col('ITEM_HIERARCHY_3'),
    )
    .distinct()
)

osa = (
    osa_raw
    .join(item, on=['ITEM_CODE'], how='left')
    .filter(F.col('ITEM_HIERARCHY_2').is_not_null())
    .cache_result()
)

feedback = (
    session.table(FEEDBACK_TABLE)
    .filter(F.col('DATE') >= LAST_WEEK_START)
    .filter(F.col('DATE') <= THIS_WEEK_END)
    .select(
        F.col('STORE_ID'),
        F.col('SITE_NAME'),
        F.col('ALERT_TYPE_NAME'),
        F.col('DATE'),
        F.col('INTERVENTION_ID'),
        F.col('TICK_AND_FLICK_FLAG'),
    )
    .cache_result()
)

markdown_log = (
    session.table(MARKDOWN_LOG_TABLE)
    .filter(F.col('DATE') >= WMD_LOOKBACK)
    .filter(F.col('DATE') <= THIS_WEEK_END)
    .cache_result()
)

markdown_sales = (
    session.table(MARKDOWN_SALES_TABLE)
    .filter(F.col('DATE') >= WMD_LOOKBACK)
    .filter(F.col('DATE') <= THIS_WEEK_END)
    .cache_result()
)

wastage = (
    session.table(WASTAGE_TABLE)
    .filter(F.col('DATE') >= WMD_LOOKBACK)
    .filter(F.col('DATE') <= THIS_WEEK_END)
    .cache_result()
)

# WMD hierarchy - uses UPC (ITEM_ID) as join key, different from OSA hierarchy
wmd_item_hierarchy = (
    session.table(WMD_HIERARCHY_TABLE)
    .select(
        F.col('UPC'),        # UPC for joining with WMD fact tables
        F.col('ITEM_DESCRIPTION'),
        F.col('ITEM_HIERARCHY_1'),
        F.col('ITEM_HIERARCHY_2'),
        F.col('ITEM_HIERARCHY_3'),
    )
    .distinct()
    .cache_result()
)

store_hierarchy = (
    session.table(STORE_HIER_TABLE)
    .cache_result()
)

print(f'\nOSA rows loaded & cached      : {osa.count():,}')
print(f'Feedback rows loaded & cached : {feedback.count():,}')
print(f'\nMarkdown log rows  : {markdown_log.count():,}')
print(f'Markdown sales rows: {markdown_sales.count():,}')
print(f'Wastage rows       : {wastage.count():,}')
print(f'WMD hierarchy rows : {wmd_item_hierarchy.count():,}')
print(f'Store hierarchy rows: {store_hierarchy.count():,}')

OSA table      : SNOWFLAKE_LEARNING_DB.STORE_INSIGHT.OSA_V1
Hierarchy table: SNOWFLAKE_LEARNING_DB.STORE_INSIGHT.HIERARCHY_V1
Feedback table : SNOWFLAKE_LEARNING_DB.STORE_INSIGHT.FEEDBACK_V1
Markdown log   : SNOWFLAKE_LEARNING_DB.STORE_INSIGHT.MARKDOWN_LOG_V1
Markdown sales : SNOWFLAKE_LEARNING_DB.STORE_INSIGHT.MARKDOWN_SALES_V1
Wastage        : SNOWFLAKE_LEARNING_DB.STORE_INSIGHT.WASTAGE_V1
WMD hierarchy  : SNOWFLAKE_LEARNING_DB.STORE_INSIGHT.WMD_HIERARCHY_V1
Store hierarchy: SNOWFLAKE_LEARNING_DB.STORE_INSIGHT.STORE_HIERARCHY_V1

OSA rows loaded & cached      : 70,576,311
Feedback rows loaded & cached : 19,861

Markdown log rows  : 138,482
Markdown sales rows: 6,316,947
Wastage rows       : 53,646
WMD hierarchy rows : 199,565
Store hierarchy rows: 173


## Stage 1 — Anomaly Detection:  Runs all registered detectors (OSA, Compliance, Markdown, Waste, SKU) then merges results.

In [5]:
from store_insight.pipeline.stages._1_detect.detect import detect_signals
from store_insight.pipeline.pipeline.config import DetectorConfig

det_cfg = DetectorConfig.from_config(cfg.detector_config.model_dump() if cfg.detector_config else {})

data = osa.filter(F.col('STORE_ID') == QC_STORE_ID) if QC_STORE_ID else osa

signals = detect_signals(
    data,
    last_week_start=LAST_WEEK_START,
    last_week_end=LAST_WEEK_END,
    this_week_start=THIS_WEEK_START,
    this_week_end=THIS_WEEK_END,
    lookback_start=LOOKBACK_START,
    feedback=feedback,
    markdown_log=markdown_log,
    markdown_sales=markdown_sales,
    wastage=wastage,
    item_hierarchy=wmd_item_hierarchy,
    store_hierarchy=store_hierarchy,
    config=det_cfg,
    cfg=cfg,
)

print(f'Total merged signals: {len(signals)}')

[1_detect] TICK_AND_FLICK=14 (chain avg 32.4%)
[1_detect] MARKDOWN_OVERRIDE VS_CHAIN=10 (chain avg 2.6%)
[1_detect] MARKDOWN_OVERRIDE WOW=10
[1_detect] MARKDOWN_SELLTHROUGH VS_CHAIN=1 (chain avg 97.5%)
[1_detect] MARKDOWN_SELLTHROUGH WOW=1
[1_detect] MARKDOWN_SKU EXCESSIVE_OVERRIDE=0 (lookback=4w, min_rate=90%)
[1_detect] MARKDOWN_SKU REPEAT_REQUEST=2676 (lookback=4w, min_weeks=3, min_requests=8)
[1_detect] WASTE VS_CHAIN=33 (chain avg 28.0%)
[1_detect] WASTE WOW=38
[1_detect] 3080 signals — TICK_AND_FLICK=14, VS_CHAIN=33, WOW=38, EXCESSIVE_OVERRIDE=0, REPEAT_REQUEST=2676, REPEAT=122
Total merged signals: 3080


In [13]:
[s for s in signals if s['signal_type'] == 'MARKDOWN_SKU']

[{'signal_type': 'MARKDOWN_SKU',
  'merge_key': ('MARKDOWN_SKU', 507, 5000128942546),
  'store_id': 507,
  'site_name': 'Framlingham Supermarket',
  'item_id': 5000128942546,
  'item_description': 'CP WHOLEMEAL FARMHOUSE   ',
  'department': 'BREAD',
  'trigger_types': ['REPEAT_REQUEST'],
  'markdown_request_count': 9,
  'override_count': None,
  'lookback_weeks': 4,
  'avg_markdown_pct': 59.89},
 {'signal_type': 'MARKDOWN_SKU',
  'merge_key': ('MARKDOWN_SKU', 232, 5000128942546),
  'store_id': 232,
  'site_name': 'Wivenhoe',
  'item_id': 5000128942546,
  'item_description': 'CP WHOLEMEAL FARMHOUSE   ',
  'department': 'BREAD',
  'trigger_types': ['REPEAT_REQUEST'],
  'markdown_request_count': 12,
  'override_count': None,
  'lookback_weeks': 4,
  'avg_markdown_pct': 55.0},
 {'signal_type': 'MARKDOWN_SKU',
  'merge_key': ('MARKDOWN_SKU', 270, 5000128942546),
  'store_id': 270,
  'site_name': 'Burnham Supermarket',
  'item_id': 5000128942546,
  'item_description': 'CP WHOLEMEAL FARMHOUS

### Inspect — Signal Breakdown by Type

In [6]:
from collections import Counter

# ── Breakdown by signal_type ──────────────────────────────────────────────────
type_counts = Counter(s['signal_type'] for s in signals)
print('Signals by type:')
for stype, count in sorted(type_counts.items()):
    print(f'  {stype:<25} {count}')

# ── Trigger breakdown within each signal type ─────────────────────────────────
print()
for stype in sorted(type_counts):
    type_sigs = [s for s in signals if s['signal_type'] == stype]
    trigger_counts = Counter(tuple(sorted(s['trigger_types'])) for s in type_sigs)
    print(f'{stype} ({len(type_sigs)} signals):')
    for triggers, count in sorted(trigger_counts.items()):
        print(f'  {"+".join(triggers):<35} {count}')
    print()

Signals by type:
  COMPLIANCE                14
  MARKDOWN_OVERRIDE         12
  MARKDOWN_SELLTHROUGH      1
  MARKDOWN_SKU              2676
  OSA                       326
  WASTE                     51

COMPLIANCE (14 signals):
  TICK_AND_FLICK                      14

MARKDOWN_OVERRIDE (12 signals):
  VS_CHAIN                            2
  VS_CHAIN+WOW                        8
  WOW                                 2

MARKDOWN_SELLTHROUGH (1 signals):
  VS_CHAIN+WOW                        1

MARKDOWN_SKU (2676 signals):
  REPEAT_REQUEST                      2676

OSA (326 signals):
  REPEAT                              87
  REPEAT+VS_CHAIN                     5
  REPEAT+VS_CHAIN+WOW                 21
  REPEAT+WOW                          9
  VS_CHAIN                            27
  VS_CHAIN+WOW                        83
  WOW                                 94

WASTE (51 signals):
  VS_CHAIN                            13
  VS_CHAIN+WOW                        20
  WOW              

## Stage 2 — Scoring: Assigns each signal a composite score (or raw projected_cost if composite is disabled). Signals are sorted by score; a top-N filter can be applied before enrichment.

In [ ]:
import copy
from store_insight.pipeline.stages._2_score.score import score_all, apply_top_n_filter

# Build a scoring_cfg dict that merges the QC overrides with whatever is in cfg.scoring
_base_scoring = cfg.scoring.model_dump() if cfg.scoring else {}
scoring_cfg = {
    **_base_scoring,
    **(
        {
            'composite_weights': COMPOSITE_WEIGHTS,
            'cost_ceilings':     COST_CEILINGS,
            'benchmark_ceilings': BENCHMARK_CEILINGS,
            'wow_ceilings':      WOW_CEILINGS,
        }
        if COMPOSITE_WEIGHTS else {}
    ),
}

# Work on a shallow copy so re-running this cell is safe
scored = [copy.copy(s) for s in signals]

score_all(
    scored,
    this_week_start=THIS_WEEK_START,
    this_week_end=THIS_WEEK_END,
    osa=osa,
    markdown_log=markdown_log,
    markdown_sales=markdown_sales,
    score_weights=cfg.scoring.score_weights if cfg.scoring else None,
    scoring_cfg=scoring_cfg if COMPOSITE_WEIGHTS else None,
)

scored.sort(key=lambda s: s.get('score', 0), reverse=True)

print(f'{len(scored)} signals scored')
print(f'Score range: {scored[-1]["score"]:.4f} – {scored[0]["score"]:.4f}')
print(f'Top 5 by score:')
for s in scored[:5]:
    print(f'  store={s["store_id"]:>4}  type={s["signal_type"]:<22}  triggers={"+".join(s["trigger_types"]):<35}  score={s["score"]:.4f}')

In [ ]:
### Inspect — Composite Dimension Breakdown (top 20 signals)
# Shows each dimension's raw score alongside the final composite score.
# Only meaningful when COMPOSITE_WEIGHTS is set.

from store_insight.pipeline.stages._2_score.dimensions import (
    financial_severity, vs_control_benchmark, vs_historical_benchmark,
    thematic, historical_recency, special_cases,
)
from store_insight.pipeline.stages._2_score import composite as _composite

if not COMPOSITE_WEIGHTS:
    print('Composite scoring is disabled — set COMPOSITE_WEIGHTS in Cell 1 to see dimension breakdown.')
else:
    rows = []
    for s in scored[:20]:
        stype        = s.get('signal_type', '')
        cost         = s.get('score', 0)   # score already IS composite; reconstruct projected_cost
        # Re-derive projected_cost from composite formula (approx — purely for display)
        fs  = financial_severity.score(s, cost, COST_CEILINGS.get(stype, 0))
        vcb = vs_control_benchmark.score(s, BENCHMARK_CEILINGS.get(stype, 0))
        vhb = vs_historical_benchmark.score(s, WOW_CEILINGS.get(stype, 0))
        th  = thematic.score(s)
        hr  = historical_recency.score(s)
        sc  = special_cases.score(s)
        rows.append({
            'store': s['store_id'],
            'type':  stype,
            'triggers': '+'.join(s.get('trigger_types', [])),
            'fin_sev':  round(fs,  3),
            'vs_ctrl':  round(vcb, 3),
            'vs_hist':  round(vhb, 3),
            'thematic': round(th,  3),
            'recency':  round(hr,  3),
            'special':  round(sc,  3),
            'composite': round(s['score'], 4),
        })

    # Pretty-print as a table
    hdr = f"{'store':>5}  {'type':<22}  {'triggers':<30}  {'fin':>5}  {'ctrl':>5}  {'hist':>5}  {'them':>5}  {'rec':>5}  {'spec':>5}  {'score':>7}"
    print(hdr)
    print('-' * len(hdr))
    for r in rows:
        print(
            f"{r['store']:>5}  {r['type']:<22}  {r['triggers']:<30}  "
            f"{r['fin_sev']:>5.3f}  {r['vs_ctrl']:>5.3f}  {r['vs_hist']:>5.3f}  "
            f"{r['thematic']:>5.3f}  {r['recency']:>5.3f}  {r['special']:>5.3f}  "
            f"{r['composite']:>7.4f}"
        )

## Stage 3 — Fact Gathering: Enriches scored signals with department breakdowns, drilldowns, day-of-week, chain drift, history, projections.

In [ ]:
from store_insight.pipeline.stages._3_facts.facts import enrich_signals

enriched = enrich_signals(
    scored,
    osa,
    last_week_start=LAST_WEEK_START,
    last_week_end=LAST_WEEK_END,
    this_week_start=THIS_WEEK_START,
    this_week_end=THIS_WEEK_END,
    lookback_start=LOOKBACK_START,
    projection_weeks=PROJECTION_WEEKS,
    drilldown_top_n=DRILLDOWN_TOP_N,
    feedback=feedback,
    markdown_log=markdown_log,
    markdown_sales=markdown_sales,
    wastage=wastage,
    item_hierarchy=wmd_item_hierarchy,
    cfg=cfg,
)

print(f'{len(enriched)} signals enriched')

[2_facts] 5 signals enriched
5 signals enriched


In [ ]:
### Inspect — one enriched signal per signal type
from collections import defaultdict

by_type = defaultdict(list)
for s in enriched:
    by_type[s['signal_type']].append(s)

for stype, sigs in sorted(by_type.items()):
    s = sigs[0]
    facts_keys = list(s.get('facts', {}).keys())
    print(f'{stype} (store={s["store_id"]}, score={s.get("score", "?"):.4f})')
    print(f'  facts keys : {facts_keys}')
    print()

[{'signal_type': 'MARKDOWN_SKU',
  'merge_key': ('MARKDOWN_SKU', 242, 202320000009),
  'store_id': 242,
  'site_name': 'Fronks Road',
  'item_id': 202320000009,
  'item_description': '',
  'department': '',
  'trigger_types': ['EXCESSIVE_OVERRIDE'],
  'markdown_request_count': 6,
  'override_count': 3,
  'lookback_weeks': 4,
  'avg_markdown_pct': 50.5,
  'facts': {'sku_markdown_history': [{'week_start': '2025-06-02 00:00:00',
     'request_count': 6,
     'override_count': 3,
     'avg_markdown_pct': 50.5}]}},
 {'signal_type': 'MARKDOWN_SKU',
  'merge_key': ('MARKDOWN_SKU', 633, 5000128104494),
  'store_id': 633,
  'site_name': 'Bishops Stortford',
  'item_id': 5000128104494,
  'item_description': '',
  'department': '',
  'trigger_types': ['EXCESSIVE_OVERRIDE'],
  'markdown_request_count': 6,
  'override_count': 3,
  'lookback_weeks': 4,
  'avg_markdown_pct': 65.33,
  'facts': {'sku_markdown_history': [{'week_start': '2025-05-19 00:00:00',
     'request_count': 6,
     'override_count

### Inspect — Facts QC Checks per Signal Type

In [ ]:
from store_insight.pipeline.stages._4_investigate.investigate import investigate
from store_insight.pipeline.stages._2_score.score import short_long_term_label

narratives = [investigate(sig, session=session, cfg=cfg) for sig in enriched]

REQUIRED_FIELDS = ['issue', 'explanation', 'cost_if_not_fixed', 'recommended_action']
blank_fields = [
    (enriched[i]['signal_type'], enriched[i]['store_id'], k)
    for i, nar in enumerate(narratives)
    for k in REQUIRED_FIELDS if not nar.get(k)
]

print(f'{len(narratives)} narratives generated')
if blank_fields:
    print(f'WARNING — {len(blank_fields)} blank fields:')
    for b in blank_fields[:10]:
        print(f'  type={b[0]}  store={b[1]}  missing={b[2]}')
else:
    print('All four fields populated for every narrative.')

KeyError: 'this_week_lsv'

In [ ]:
## Stage 4 — Full Card Preview: one complete alert card per signal type, including short_long_term_label.

[{'issue': 'WADGATE ROAD: 59.2% of alerts tick-and-flicked this week',
  'explanation': '45 of 76 alerts worked this week were tick-and-flick (59.2%). This is a +25.4pp increase from last week and +23.8pp above the chain average. Worst alert type by tick-and-flick rate: Availability (59.2%).',
  'cost_if_not_fixed': 'Unresolved alerts mean availability issues go unaddressed, increasing the risk of lost sales across affected departments.',
  'recommended_action': 'Review alert working practice with your team at WADGATE ROAD. Spot-check the Availability alert type and ensure all alerts are genuinely investigated. Consider coaching your team to improve compliance.'},
 {'issue': 'WIMPOLE ROAD: 59.6% of alerts tick-and-flicked this week',
  'explanation': '65 of 109 alerts worked this week were tick-and-flick (59.6%). This is a +16.9pp increase from last week and 24.2pp above the chain average. Worst alert type by tick-and-flick rate: Availability (59.6%).',
  'cost_if_not_fixed': 'Unresolv

In [ ]:
from store_insight.pipeline.pipeline.pipeline import _build_card

by_type = {}
for sig, nar in zip(enriched, narratives):
    stype = sig['signal_type']
    if stype not in by_type:
        label = short_long_term_label(sig)
        card  = _build_card(sig, nar, sig['score'], cfg=cfg, label=label)
        by_type[stype] = card

for stype, card in sorted(by_type.items()):
    print(f'═══ {stype} ═══')
    for k, v in card.items():
        if k in ('signal_json', 'facts_json', 'llm_prompt'):
            continue
        print(f'  {k:<22}: {v}')
    print()